# Week 3 — Sequence Builder: `(N, 32, F)` 张量构造

**目标**：
- 把 **合成 parquet** 按 `user_id + ts` 排序、滑窗 L=32 → `(N, 32, F_synth)`
- 把 **Kaggle creditcardfraud CSV**（无 userId）用 hash 伪造 `user_id`，同样滑窗
- **按时间切分**：前 70% train / 中 15% val / 后 15% test（**绝不**随机切）
- 封装 `torch.utils.data.Dataset` 子类并保存张量到 `data/processed/`

**产出**：
- `data/processed/synth_{train,val,test}.pt`
- `data/processed/kaggle_{train,val,test}.pt`
- 验证输出：无时间泄露、形状正确、标签平衡统计

In [ ]:
# ── Bootstrap (Colab + local) ──────────────────────────────────────
# Env detect → Drive mount → Kaggle creds → reproducibility
import os, sys, pathlib, random

IN_COLAB = 'google.colab' in sys.modules
DRIVE_FOLDER = 'LLM/transformer-anomaly'  # edit if your Drive subfolder differs

if IN_COLAB:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    PROJECT_ROOT = pathlib.Path(f'/content/drive/MyDrive/{DRIVE_FOLDER}')
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    for sub in ('data', 'models', 'checkpoints'):
        (PROJECT_ROOT / sub).mkdir(exist_ok=True)
    os.chdir(PROJECT_ROOT)

    # Kaggle credentials via Colab Secrets (left sidebar 🔑 icon).
    # Add KAGGLE_USERNAME and KAGGLE_KEY there; never upload kaggle.json.
    try:
        from google.colab import userdata
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
    except Exception as e:
        print(f'[bootstrap] Kaggle secrets not configured: {e}')
else:
    PROJECT_ROOT = pathlib.Path.cwd()

# Reproducibility
import numpy as np, torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[bootstrap] env={"colab" if IN_COLAB else "local"}  device={device}')
print(f'[bootstrap] project_root={PROJECT_ROOT}')


## 1. 加载两个数据源

In [ ]:
import numpy as np, pandas as pd, torch
from pathlib import Path

synth_path = PROJECT_ROOT / 'data' / 'synth' / 'transactions.parquet'
kaggle_csv = PROJECT_ROOT / 'data' / 'creditcard.csv'

# 合成数据（若缺失，提示先跑 03_synth_data.ipynb）
if not synth_path.exists():
    raise FileNotFoundError(f'run 03_synth_data.ipynb first to produce {synth_path}')

# Kaggle（若缺失尝试重新下载）
if not kaggle_csv.exists():
    (PROJECT_ROOT / 'data').mkdir(exist_ok=True)
    !pip install -q kaggle
    !kaggle datasets download -d mlg-ulb/creditcardfraud -p {PROJECT_ROOT / 'data'} --unzip

df_synth = pd.read_parquet(synth_path)
df_kaggle = pd.read_csv(kaggle_csv)
print('synth :', df_synth.shape, '  anomaly%:', df_synth.is_anomaly.mean()*100)
print('kaggle:', df_kaggle.shape, '  fraud%  :', df_kaggle.Class.mean()*100)

## 2. 特征工程

### 合成数据
- 数值：`amount`, `hour` (cos/sin), `lat`, `lng`
- 类别 → 整数 id：`merchant_cat`, `device_id`（整数就够了，W4 LSTM 里可以走 Embedding 或直接 one-hot）
- 我们先把全部转成稠密 float：类别做 label encoding → 归一化到 [0, 1] / `n_cat`

### Kaggle
- 已经 PCA 脱敏（V1–V28），只需对 `Time / Amount` 标准化即可。
- **模拟 user_id**：`(Time_bucket × 97 + round(Amount) × 53) mod 500` → 500 个伪用户。目的是让同一用户序列里尽量"风格相似"，虽然不完美但能测通序列管线。

In [ ]:
# ─── 合成数据特征 ───
df_synth = df_synth.sort_values(['user_id', 'ts']).reset_index(drop=True)
df_synth['hour'] = df_synth.ts.dt.hour + df_synth.ts.dt.minute / 60.0
df_synth['hour_sin'] = np.sin(2 * np.pi * df_synth['hour'] / 24)
df_synth['hour_cos'] = np.cos(2 * np.pi * df_synth['hour'] / 24)
df_synth['amount_log'] = np.log1p(df_synth['amount'])

# label encode
for col in ['merchant_cat', 'device_id']:
    df_synth[col + '_id'] = df_synth[col].astype('category').cat.codes.astype('int32')

SYNTH_FEATS = ['amount_log', 'hour_sin', 'hour_cos', 'lat', 'lng',
               'merchant_cat_id', 'device_id_id']
print('synth features:', SYNTH_FEATS)

# 标准化（只用 train 统计量，这里先占位，切分后再 fit）
df_synth[['ts_unix']] = df_synth[['ts']].astype('int64') // 10**9
df_synth.head(3)

In [ ]:
# ─── Kaggle 特征 + 伪 user_id ───
df_kaggle = df_kaggle.sort_values('Time').reset_index(drop=True)
time_bucket = (df_kaggle['Time'] // 60).astype('int64')   # 每分钟一个桶
amt_int = df_kaggle['Amount'].round().astype('int64')
df_kaggle['user_id'] = ((time_bucket * 97 + amt_int * 53) % 500).astype('int32')
df_kaggle['ts_unix'] = df_kaggle['Time'].astype('int64')
df_kaggle['is_anomaly'] = df_kaggle['Class'].astype('int32')

KAGGLE_FEATS = [f'V{i}' for i in range(1, 29)] + ['Amount']
print('kaggle features:', len(KAGGLE_FEATS))
print('pseudo-user counts:', df_kaggle.user_id.nunique())
df_kaggle[['user_id','ts_unix','Amount','is_anomaly']].head(3)

## 3. 按时间切分

**关键**：先挑一个全局时间分位（不是按用户），把数据切成三段：
- train: `ts < q70`
- val:   `q70 ≤ ts < q85`
- test:  `ts ≥ q85`

然后在每段内部按 `(user_id, ts)` 排序 → 滑窗。这样保证**未来的交易绝不会出现在训练集**，消除时间泄露。

In [ ]:
def split_by_time(df, ts_col='ts_unix'):
    q70 = df[ts_col].quantile(0.70)
    q85 = df[ts_col].quantile(0.85)
    return (df[df[ts_col] <  q70].copy(),
            df[(df[ts_col] >= q70) & (df[ts_col] < q85)].copy(),
            df[df[ts_col] >= q85].copy())

syn_tr, syn_va, syn_te = split_by_time(df_synth)
kg_tr,  kg_va,  kg_te  = split_by_time(df_kaggle)
for name, parts in [('synth', (syn_tr, syn_va, syn_te)),
                    ('kaggle', (kg_tr, kg_va, kg_te))]:
    tr, va, te = parts
    print(f'{name}: train={len(tr)} val={len(va)} test={len(te)}  '
          f'anomaly% train={tr.is_anomaly.mean()*100:.3f} '
          f'val={va.is_anomaly.mean()*100:.3f} test={te.is_anomaly.mean()*100:.3f}')

## 4. 滑动窗口 → `(N, 32, F)`

规则：
- 对每个用户按时间升序排列
- 滑窗长度 `L=32`，步长 1
- 窗口标签 = 窗口**最后一笔**的 `is_anomaly`（业务语义："给过去 32 笔，判断最后这一笔是否异常"）
- 用户交易数 < L 则整段补 0 并用 padding mask（这里为简单起见直接丢弃 < L 的用户）

In [ ]:
L = 32

def build_sequences(df, feat_cols, L=32):
    xs, ys = [], []
    for uid, g in df.sort_values(['user_id', 'ts_unix']).groupby('user_id', sort=False):
        arr = g[feat_cols].to_numpy(dtype='float32')
        lbl = g['is_anomaly'].to_numpy(dtype='int64')
        if len(arr) < L:
            continue
        for i in range(L, len(arr) + 1):
            xs.append(arr[i - L:i])
            ys.append(lbl[i - 1])
    if not xs:
        return np.zeros((0, L, len(feat_cols)), dtype='float32'), np.zeros((0,), dtype='int64')
    return np.stack(xs), np.array(ys, dtype='int64')

# 合成
Xtr_s, ytr_s = build_sequences(syn_tr, SYNTH_FEATS, L)
Xva_s, yva_s = build_sequences(syn_va, SYNTH_FEATS, L)
Xte_s, yte_s = build_sequences(syn_te, SYNTH_FEATS, L)
print('synth  train:', Xtr_s.shape, '  anomaly%:', ytr_s.mean()*100)
print('synth  val  :', Xva_s.shape, '  anomaly%:', yva_s.mean()*100)
print('synth  test :', Xte_s.shape, '  anomaly%:', yte_s.mean()*100)

# Kaggle
Xtr_k, ytr_k = build_sequences(kg_tr, KAGGLE_FEATS, L)
Xva_k, yva_k = build_sequences(kg_va, KAGGLE_FEATS, L)
Xte_k, yte_k = build_sequences(kg_te, KAGGLE_FEATS, L)
print('kaggle train:', Xtr_k.shape, '  fraud%:', ytr_k.mean()*100)
print('kaggle val  :', Xva_k.shape, '  fraud%:', yva_k.mean()*100)
print('kaggle test :', Xte_k.shape, '  fraud%:', yte_k.mean()*100)

## 5. 标准化（只用训练集统计量）

In [ ]:
def fit_standardize(X):
    # X: (N, L, F) → mean/std over (N, L)
    mean = X.reshape(-1, X.shape[-1]).mean(0)
    std  = X.reshape(-1, X.shape[-1]).std(0) + 1e-6
    return mean.astype('float32'), std.astype('float32')

def apply_std(X, mean, std):
    return ((X - mean) / std).astype('float32')

m_s, s_s = fit_standardize(Xtr_s)
Xtr_s = apply_std(Xtr_s, m_s, s_s); Xva_s = apply_std(Xva_s, m_s, s_s); Xte_s = apply_std(Xte_s, m_s, s_s)

m_k, s_k = fit_standardize(Xtr_k)
Xtr_k = apply_std(Xtr_k, m_k, s_k); Xva_k = apply_std(Xva_k, m_k, s_k); Xte_k = apply_std(Xte_k, m_k, s_k)

print('synth mean[0:3]:', m_s[:3], '  std[0:3]:', s_s[:3])
print('kaggle mean[0:3]:', m_k[:3], '  std[0:3]:', s_k[:3])

## 6. 泄露自检

1. **时间泄露**：train 的最大时间 < val 的最小时间？val max < test min？
2. **标签漂移**：三段的 anomaly% 差异在合理范围（欺诈本身偏斜是正常的，不做校正）。

In [ ]:
def leak_check(name, tr_df, va_df, te_df):
    tr_max = tr_df.ts_unix.max(); va_min = va_df.ts_unix.min()
    va_max = va_df.ts_unix.max(); te_min = te_df.ts_unix.min()
    ok1 = tr_max <= va_min
    ok2 = va_max <= te_min
    print(f'[{name}] train.max={tr_max}  val.min={va_min}  {"OK" if ok1 else "LEAK"}')
    print(f'[{name}] val.max  ={va_max}  test.min={te_min} {"OK" if ok2 else "LEAK"}')
    assert ok1 and ok2, f'{name} time leak!'

leak_check('synth',  syn_tr, syn_va, syn_te)
leak_check('kaggle', kg_tr,  kg_va,  kg_te)

## 7. `torch.utils.data.Dataset` 封装 + 保存

In [ ]:
from torch.utils.data import Dataset

class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)         # (N, L, F)
        self.y = torch.from_numpy(y).long()  # (N,)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

out_dir = PROJECT_ROOT / 'data' / 'processed'
out_dir.mkdir(parents=True, exist_ok=True)

def dump(prefix, splits):
    for name, (X, y) in splits.items():
        torch.save({'X': torch.from_numpy(X), 'y': torch.from_numpy(y).long()},
                   out_dir / f'{prefix}_{name}.pt')
        print(f'saved {prefix}_{name}.pt  X={X.shape}  y={y.shape}  pos={int(y.sum())}')

dump('synth', {'train': (Xtr_s, ytr_s), 'val': (Xva_s, yva_s), 'test': (Xte_s, yte_s)})
dump('kaggle', {'train': (Xtr_k, ytr_k), 'val': (Xva_k, yva_k), 'test': (Xte_k, yte_k)})

## 8. Dataset smoke test

In [ ]:
ds = SeqDataset(Xva_s, yva_s)
print('len:', len(ds), '  sample x:', ds[0][0].shape, '  y:', ds[0][1].item())

# 快速 DataLoader
from torch.utils.data import DataLoader
dl = DataLoader(ds, batch_size=32, shuffle=False)
xb, yb = next(iter(dl))
print('batch x:', xb.shape, '  y:', yb.shape, '  y dtype:', yb.dtype)

## 9. 下一步

Week 4 的 `04_lstm_baseline.ipynb` 会 `torch.load('data/processed/synth_train.pt')` 直接喂给 LSTM，
达成：
- 合成数据 AUC-PR > 0.95
- Kaggle 序列 AUC-PR > Week 2 的 MLP baseline

如果本 notebook 丢了上面的 .pt 文件，W4 的 notebook 里也内置了"从头重建"的 fallback 逻辑，保证**自包含可跑**。